# Insurance Claim Fraud Risk Triage Multi-Agent System


## Project pipeline
`preprocess-data → train-models → evaluate-models → select-model → run-inference → generate-review-checklist → generate-business-recommendation`


In [ ]:
# Data: this repo does not ship the dataset. Download the public 'Auto Insurance
# Claims Data' dataset (see README - Data) and place insurance_claims.csv under data/.
from pathlib import Path
DATA_PATH = Path("data/insurance_claims.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Download the public 'Auto Insurance Claims Data' "
        "dataset (kaggle.com/datasets/buntyshah/auto-insurance-claims-data) and "
        "place insurance_claims.csv under data/."
    )
print(f"Using dataset at {DATA_PATH}")


In [ ]:
# Dependencies are listed in requirements.txt:
#   pip install -r requirements.txt
# This cell is kept as an optional convenience installer.
import sys, subprocess, pkgutil

def ensure_install(package_spec: str):
    pkg_name = package_spec.split("==")[0].split(">=")[0].split("[")[0].replace("-", "_")
    if pkgutil.find_loader(pkg_name) is None:
        print(f"Installing {package_spec} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_spec])

for pkg in [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "joblib",
    "catboost",
    "lightgbm",
    "langgraph",
    "openai",
    "pydantic",
    "gradio"
]:
    ensure_install(pkg)

print("Package setup complete.")


In [ ]:

import os
import json
import warnings
from pathlib import Path
from typing import TypedDict, Dict, Any, List

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pydantic import Field, create_model
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    confusion_matrix, roc_curve
)

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from langgraph.graph import StateGraph, END
from openai import OpenAI

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
sns.set_context("talk")

PROJECT_DIR = Path("data")
DATA_PATH = PROJECT_DIR / "insurance_claims.csv"
ARTIFACT_DIR = PROJECT_DIR / "../models"
SKILLS_DIR = PROJECT_DIR / "skills"
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)
SKILLS_DIR.mkdir(exist_ok=True, parents=True)

print(f"Using DATA_PATH={DATA_PATH}")
print(f"Artifacts directory: {ARTIFACT_DIR.resolve()}")


In [ ]:

# Load and inspect the dataset
df = pd.read_csv(DATA_PATH)
print(df.shape)
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())
print("\nTarget distribution:")
display(df["fraud_reported"].value_counts(dropna=False))

In [ ]:

# Basic data quality checks
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
display(missing.to_frame("missing_count").head(10))

# Visualise target imbalance
target_counts = df["fraud_reported"].value_counts()
plt.figure()
plt.bar(target_counts.index.astype(str), target_counts.values)
plt.title("Target distribution: fraud_reported")
plt.xlabel("fraud_reported")
plt.ylabel("count")
plt.show()

## Agent state definition

A typed shared state that carries data, model artifacts, evaluation results, business rules, and downstream outputs

In [ ]:
class PipelineState(TypedDict, total=False):
    raw_data_path: str
    artifact_dir: str
    skills_dir: str
    demo_case_index: int

    dataset_profile: Dict[str, Any]
    target_column: str
    split_paths: Dict[str, str]
    raw_split_paths: Dict[str, str]
    model_split_paths: Dict[str, Dict[str, str]]
    model_preprocessor_paths: Dict[str, str]

    feature_columns: List[str]
    numeric_columns: List[str]
    categorical_columns: List[str]
    nominal_categorical_columns: List[str]
    ordinal_categorical_columns: List[str]
    ordinal_category_orders: Dict[str, List[str]]
    nominal_category_values: Dict[str, List[str]]

    trained_model_paths: Dict[str, str]
    trained_model_families: Dict[str, str]
    eval_metrics: Dict[str, Dict[str, float]]
    threshold_calibrations: Dict[str, Dict[str, Any]]
    evaluation_summary_table: List[Dict[str, Any]]
    figures: Dict[str, Any]
    threshold_policy: Dict[str, Any]

    selected_model_id: str
    selected_model_path: str
    selected_model_family: str
    selected_model_reasoning: Dict[str, Any]

    inference_input: Dict[str, Any]
    inference_result_raw: Dict[str, Any]
    review_checklist_output: Dict[str, Any]
    business_output: Dict[str, Any]

    logs: List[str]
    errors: List[str]


## OpenAI client setup


In [ ]:
from functools import lru_cache


def get_openai_api_key() -> str:
    api_key = os.environ.get("OPENAI_API_KEY")
    if api_key:
        return api_key

    try:
        from google.colab import userdata  # type: ignore
        api_key = userdata.get("OPENAI_API_KEY")
        if api_key:
            return api_key
    except Exception:
        pass

    raise RuntimeError(
        "OPENAI_API_KEY not found. Add it to environment variables or Colab Secrets "
        "before running select-model or downstream LLM nodes."
    )


@lru_cache(maxsize=1)
def get_openai_client() -> OpenAI:
    return OpenAI(api_key=get_openai_api_key())



## Shared preprocessing and model-specific dataset construction

- shared dataset cleaning / feature engineering
- identification of ordinal vs nominal categorical variables
- model-specific dataset construction for logistic regression, catboost, and lightgbm


In [ ]:
TARGET_COLUMN = "fraud_reported"
DROP_COLUMNS_BEFORE_MODELING = [
    "policy_number",
    "insured_zip",
    "incident_location",
    "_c39",
]
MISSING_CATEGORY_TOKENS = {"?", "", "MISSING"}
SEVERITY_ORDER = ["Trivial Damage", "Minor Damage", "Major Damage", "Total Loss"]


def extract_target_series(df_in: pd.DataFrame) -> pd.Series:
    target = df_in[TARGET_COLUMN].copy()
    if target.dtype == object:
        target = target.map({"Y": 1, "N": 0})
    return target.astype(int)


def build_feature_matrix_from_raw(X_raw: pd.DataFrame) -> pd.DataFrame:
    X = X_raw.copy()

    drop_cols = [c for c in DROP_COLUMNS_BEFORE_MODELING if c in X.columns]
    if drop_cols:
        X = X.drop(columns=drop_cols)

    for date_col in ["policy_bind_date", "incident_date"]:
        if date_col in X.columns:
            X[date_col] = pd.to_datetime(X[date_col], errors="coerce")

    if {"policy_bind_date", "incident_date"}.issubset(X.columns):
        days_since_bind = (X["incident_date"] - X["policy_bind_date"]).dt.days
        X["invalid_date_order"] = ((days_since_bind < 0) & days_since_bind.notna()).astype(int)
        X["days_since_policy_bind_to_incident"] = days_since_bind.clip(lower=0)

    for date_col in ["policy_bind_date", "incident_date"]:
        if date_col in X.columns:
            X = X.drop(columns=[date_col])

    categorical_columns = X.select_dtypes(exclude=["number"]).columns.tolist()
    for col in categorical_columns:
        X[col] = X[col].replace(list(MISSING_CATEGORY_TOKENS), np.nan)

    return X


def build_base_feature_frame(df_in: pd.DataFrame):
    df_local = df_in.copy()
    y = extract_target_series(df_local)
    X_raw = df_local.drop(columns=[TARGET_COLUMN]).copy()
    X = build_feature_matrix_from_raw(X_raw)
    return X, y


def infer_ordinal_order(series: pd.Series):
    values = [str(v) for v in series.dropna().astype(str).unique().tolist()]
    if not values:
        return None

    if all("/" in v and len(v.split("/")) == 2 and all(part.isdigit() for part in v.split("/")) for v in values):
        return sorted(set(values), key=lambda v: tuple(int(part) for part in v.split("/")))

    if set(values).issubset(set(SEVERITY_ORDER)):
        return [label for label in SEVERITY_ORDER if label in set(values)]

    return None


def derive_feature_metadata(X_df: pd.DataFrame) -> Dict[str, Any]:
    feature_columns = X_df.columns.tolist()
    numeric_columns = X_df.select_dtypes(include=["number"]).columns.tolist()
    categorical_candidates = [c for c in feature_columns if c not in numeric_columns]

    ordinal_category_orders = {}
    nominal_categorical_columns = []

    for col in categorical_candidates:
        inferred_order = infer_ordinal_order(X_df[col])
        if inferred_order is not None:
            ordinal_category_orders[col] = inferred_order
        else:
            nominal_categorical_columns.append(col)

    ordinal_categorical_columns = list(ordinal_category_orders.keys())
    categorical_columns = nominal_categorical_columns + ordinal_categorical_columns

    return {
        "feature_columns": feature_columns,
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
        "nominal_categorical_columns": nominal_categorical_columns,
        "ordinal_categorical_columns": ordinal_categorical_columns,
        "ordinal_category_orders": ordinal_category_orders,
    }


def build_nominal_category_values(X_train: pd.DataFrame, nominal_categorical_columns: List[str]) -> Dict[str, List[str]]:
    nominal_category_values = {}
    for col in nominal_categorical_columns:
        categories = sorted(X_train[col].fillna("MISSING").astype(str).unique().tolist())
        if "MISSING" not in categories:
            categories.append("MISSING")
        nominal_category_values[col] = categories
    return nominal_category_values


def rebuild_raw_splits(raw_data_path: str) -> Dict[str, pd.DataFrame]:
    raw_df = pd.read_csv(raw_data_path).copy()
    y_all = extract_target_series(raw_df)
    X_all_raw = raw_df.drop(columns=[TARGET_COLUMN]).copy()

    X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
        X_all_raw, y_all, test_size=0.30, random_state=42, stratify=y_all
    )
    X_val_raw, X_test_raw, y_val, y_test = train_test_split(
        X_temp_raw, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    return {
        "X_train_raw": X_train_raw.reset_index(drop=True),
        "X_val_raw": X_val_raw.reset_index(drop=True),
        "X_test_raw": X_test_raw.reset_index(drop=True),
        "y_train": y_train.reset_index(drop=True),
        "y_val": y_val.reset_index(drop=True),
        "y_test": y_test.reset_index(drop=True),
    }


In [ ]:


def encode_ordinal_columns(X_df: pd.DataFrame, ordinal_category_orders: Dict[str, List[str]]) -> pd.DataFrame:
    X = X_df.copy()
    for col, order in ordinal_category_orders.items():
        if col in X.columns:
            codes = pd.Categorical(X[col], categories=order, ordered=True).codes.astype(float)
            X[col] = pd.Series(codes, index=X.index).replace(-1, np.nan)
    return X



def build_logreg_preprocessor(
    numeric_columns: List[str],
    ordinal_categorical_columns: List[str],
    nominal_categorical_columns: List[str],
):
    numeric_like_columns = numeric_columns + ordinal_categorical_columns
    transformers = []

    if numeric_like_columns:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("num", numeric_transformer, numeric_like_columns))

    if nominal_categorical_columns:
        nominal_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
        ])
        transformers.append(("nominal", nominal_transformer, nominal_categorical_columns))

    return ColumnTransformer(transformers=transformers)



def prepare_tree_model_frame(X_df: pd.DataFrame, feature_meta: Dict[str, Any], model_family: str) -> pd.DataFrame:
    X = encode_ordinal_columns(X_df, feature_meta["ordinal_category_orders"])

    for col in feature_meta["nominal_categorical_columns"]:
        if col not in X.columns:
            continue

        categories = feature_meta["nominal_category_values"][col]
        raw = X[col].fillna("MISSING").astype(str)
        raw = raw.where(raw.isin(categories), "MISSING")

        if model_family == "catboost":
            X[col] = raw
        elif model_family == "lightgbm":
            X[col] = pd.Categorical(raw, categories=categories, ordered=False)

    return X



def build_model_ready_datasets(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    feature_meta: Dict[str, Any],
    artifact_dir: Path,
):
    model_split_paths = {
        "logreg": {},
        "catboost": {},
        "lightgbm": {},
    }
    model_preprocessor_paths = {}

    nominal_category_values = build_nominal_category_values(X_train, feature_meta["nominal_categorical_columns"])
    feature_meta = {
        **feature_meta,
        "nominal_category_values": nominal_category_values,
    }

    X_train_ord = encode_ordinal_columns(X_train, feature_meta["ordinal_category_orders"])
    X_val_ord = encode_ordinal_columns(X_val, feature_meta["ordinal_category_orders"])
    X_test_ord = encode_ordinal_columns(X_test, feature_meta["ordinal_category_orders"])

    logreg_preprocessor = build_logreg_preprocessor(
        feature_meta["numeric_columns"],
        feature_meta["ordinal_categorical_columns"],
        feature_meta["nominal_categorical_columns"],
    )
    logreg_frames = {
        "X_train": logreg_preprocessor.fit_transform(X_train_ord),
        "X_val": logreg_preprocessor.transform(X_val_ord),
        "X_test": logreg_preprocessor.transform(X_test_ord),
    }
    logreg_preprocessor_path = artifact_dir / "logreg_preprocessor.joblib"
    joblib.dump(logreg_preprocessor, logreg_preprocessor_path)
    model_preprocessor_paths["logreg"] = str(logreg_preprocessor_path)

    for split_name, obj in logreg_frames.items():
        p = artifact_dir / f"logreg_{split_name}.joblib"
        joblib.dump(obj, p)
        model_split_paths["logreg"][split_name] = str(p)

    for model_family in ["catboost", "lightgbm"]:
        ready_frames = {
            "X_train": prepare_tree_model_frame(X_train, feature_meta, model_family),
            "X_val": prepare_tree_model_frame(X_val, feature_meta, model_family),
            "X_test": prepare_tree_model_frame(X_test, feature_meta, model_family),
        }
        for split_name, obj in ready_frames.items():
            p = artifact_dir / f"{model_family}_{split_name}.joblib"
            joblib.dump(obj, p)
            model_split_paths[model_family][split_name] = str(p)

    return feature_meta, model_split_paths, model_preprocessor_paths



def prepare_model_input_for_inference(X_df: pd.DataFrame, state: PipelineState, model_family: str):
    feature_meta = {
        "ordinal_category_orders": state["ordinal_category_orders"],
        "nominal_categorical_columns": state["nominal_categorical_columns"],
        "nominal_category_values": state["nominal_category_values"],
    }

    if model_family == "logreg":
        X_ord = encode_ordinal_columns(X_df, state["ordinal_category_orders"])
        preprocessor = joblib.load(state["model_preprocessor_paths"]["logreg"])
        return preprocessor.transform(X_ord)

    return prepare_tree_model_frame(X_df, feature_meta, model_family)



def build_candidate_models() -> Dict[str, Dict[str, Any]]:
    return {
        "logreg_baseline": {
            "family": "logreg",
            "model": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        },
        "catboost": {
            "family": "catboost",
            "model": CatBoostClassifier(
                iterations=250,
                depth=6,
                learning_rate=0.05,
                loss_function="Logloss",
                eval_metric="AUC",
                auto_class_weights="Balanced",
                random_seed=42,
                verbose=0,
            ),
        },
        "lightgbm": {
            "family": "lightgbm",
            "model": LGBMClassifier(
                n_estimators=250,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=31,
                subsample=0.9,
                colsample_bytree=0.9,
                is_unbalance=True,
                random_state=42,
                verbosity=-1,
            ),
        },
    }



def predict_positive_probability(model, X_eval) -> np.ndarray:
    prob = model.predict_proba(X_eval)
    if getattr(prob, "ndim", 1) == 2:
        prob = prob[:, 1]
    return np.asarray(prob, dtype=float)


In [ ]:
FN_COST_PROXY = 75.2
FP_COST_PROXY = 9.8
COST_DERIVED_BETA = float(np.sqrt(FN_COST_PROXY / FP_COST_PROXY))
FBETA_COST_METRIC = "fbeta_cost"


def pretty_calibration_metric(metric_name: str) -> str:
    if metric_name == FBETA_COST_METRIC:
        return f"Fβ(beta={COST_DERIVED_BETA:.2f})"
    return metric_name.upper()



def evaluate_binary_classifier(model, X_eval, y_eval: pd.Series, threshold: float) -> Dict[str, float]:
    prob = predict_positive_probability(model, X_eval)
    pred = (prob >= threshold).astype(int)

    return {
        "accuracy": float(accuracy_score(y_eval, pred)),
        "precision": float(precision_score(y_eval, pred, zero_division=0)),
        "recall": float(recall_score(y_eval, pred, zero_division=0)),
        "f1": float(f1_score(y_eval, pred, zero_division=0)),
        "fbeta_cost": float(fbeta_score(y_eval, pred, beta=COST_DERIVED_BETA, zero_division=0)),
        "fbeta_beta": float(COST_DERIVED_BETA),
        "auc_roc": float(roc_auc_score(y_eval, prob)),
        "decision_threshold": float(threshold),
    }



def calibrate_threshold(model, X_val, y_val: pd.Series, metric: str = FBETA_COST_METRIC) -> Dict[str, Any]:
    prob = predict_positive_probability(model, X_val)

    threshold_grid = np.arange(0.1, 0.95, 0.05)
    best_threshold = float(threshold_grid[0])
    best_score = -1
    threshold_results = []

    for threshold in threshold_grid:
        pred = (prob >= threshold).astype(int)

        if metric == "precision":
            score = precision_score(y_val, pred, zero_division=0)
        elif metric == "recall":
            score = recall_score(y_val, pred, zero_division=0)
        elif metric == "balanced":
            prec = precision_score(y_val, pred, zero_division=0)
            rec = recall_score(y_val, pred, zero_division=0)
            score = (prec + rec) / 2
        elif metric == FBETA_COST_METRIC:
            score = fbeta_score(y_val, pred, beta=COST_DERIVED_BETA, zero_division=0)
        else:
            score = f1_score(y_val, pred, zero_division=0)

        threshold_results.append({
            "threshold": float(threshold),
            "score": float(score),
            "metric": metric,
            "metric_label": pretty_calibration_metric(metric),
            "beta": float(COST_DERIVED_BETA) if metric == FBETA_COST_METRIC else None,
        })

        if score > best_score:
            best_score = score
            best_threshold = threshold

    return {
        "optimal_threshold": float(best_threshold),
        "optimal_score": float(best_score),
        "calibration_metric": metric,
        "calibration_metric_label": pretty_calibration_metric(metric),
        "calibration_beta": float(COST_DERIVED_BETA) if metric == FBETA_COST_METRIC else None,
        "threshold_sweep": threshold_results,
    }



In [ ]:


def generate_evaluation_figures(
    models_dict: Dict[str, Any],
    X_eval_by_family: Dict[str, Any],
    y_eval: pd.Series,
    state: PipelineState,
    threshold_calibrations: Dict[str, Dict[str, Any]],
) -> Dict[str, Any]:
    figures = {}

    for model_id, model in models_dict.items():
        model_family = state["trained_model_families"][model_id]
        X_eval = X_eval_by_family[model_family]
        prob = predict_positive_probability(model, X_eval)
        threshold = threshold_calibrations[model_id]["optimal_threshold"]
        pred = (prob >= threshold).astype(int)

        cm = confusion_matrix(y_eval, pred)
        fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cbar=False, ax=ax_cm)
        ax_cm.set_title(f"Confusion Matrix - {model_id} (threshold={threshold:.2f})")
        ax_cm.set_xlabel("Predicted")
        ax_cm.set_ylabel("Actual")
        plt.tight_layout()
        figures[f"{model_id}_confusion_matrix"] = fig_cm

        fpr, tpr, _ = roc_curve(y_eval, prob)
        fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
        ax_roc.plot(fpr, tpr, label=f"{model_id} (AUC={roc_auc_score(y_eval, prob):.3f})")
        ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray")
        ax_roc.set_xlabel("False Positive Rate")
        ax_roc.set_ylabel("True Positive Rate")
        ax_roc.set_title(f"ROC Curve - {model_id}")
        ax_roc.legend()
        plt.tight_layout()
        figures[f"{model_id}_roc_curve"] = fig_roc

    return figures


## Agent nodes

In [ ]:
def preprocess_data_node(state: PipelineState) -> Dict[str, Any]:
    df_local = pd.read_csv(state["raw_data_path"]).copy()
    artifact_dir = Path(state["artifact_dir"])
    artifact_dir.mkdir(exist_ok=True, parents=True)

    y_all = extract_target_series(df_local)
    X_all_raw = df_local.drop(columns=[TARGET_COLUMN]).copy()

    X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
        X_all_raw, y_all, test_size=0.30, random_state=42, stratify=y_all
    )
    X_val_raw, X_test_raw, y_val, y_test = train_test_split(
        X_temp_raw, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )

    X_train = build_feature_matrix_from_raw(X_train_raw.reset_index(drop=True))
    X_val = build_feature_matrix_from_raw(X_val_raw.reset_index(drop=True))
    X_test = build_feature_matrix_from_raw(X_test_raw.reset_index(drop=True))

    feature_meta = derive_feature_metadata(X_train)
    feature_meta, model_split_paths, model_preprocessor_paths = build_model_ready_datasets(
        X_train,
        X_val,
        X_test,
        feature_meta,
        artifact_dir,
    )

    split_paths = {}
    for name, obj in {
        "X_train": X_train.reset_index(drop=True),
        "X_val": X_val.reset_index(drop=True),
        "X_test": X_test.reset_index(drop=True),
        "y_train": y_train.reset_index(drop=True),
        "y_val": y_val.reset_index(drop=True),
        "y_test": y_test.reset_index(drop=True),
    }.items():
        p = artifact_dir / f"{name}.joblib"
        joblib.dump(obj, p)
        split_paths[name] = str(p)

    raw_split_paths = {}
    for name, obj in {
        "X_train_raw": X_train_raw.reset_index(drop=True),
        "X_val_raw": X_val_raw.reset_index(drop=True),
        "X_test_raw": X_test_raw.reset_index(drop=True),
    }.items():
        p = artifact_dir / f"{name}.joblib"
        joblib.dump(obj, p)
        raw_split_paths[name] = str(p)

    dataset_profile = {
        "row_count": int(len(df_local)),
        "positive_rate": float(y_all.mean()),
        "target_distribution": y_all.value_counts(normalize=True).to_dict(),
        "missing_values_top10": df_local.isna().sum().sort_values(ascending=False).head(10).to_dict(),
    }

    return {
        "dataset_profile": dataset_profile,
        "target_column": TARGET_COLUMN,
        "split_paths": split_paths,
        "raw_split_paths": raw_split_paths,
        "model_split_paths": model_split_paths,
        "model_preprocessor_paths": model_preprocessor_paths,
        **feature_meta,
        "logs": state.get("logs", []) + ["preprocess-data completed"],
    }



In [ ]:


def train_models_node(state: PipelineState) -> Dict[str, Any]:
    y_train = joblib.load(state["split_paths"]["y_train"])
    artifact_dir = Path(state["artifact_dir"])
    artifact_dir.mkdir(exist_ok=True, parents=True)

    candidates = build_candidate_models()
    trained_model_paths = {}
    trained_model_families = {}

    for model_id, candidate in candidates.items():
        model_family = candidate["family"]
        model = candidate["model"]
        X_train = joblib.load(state["model_split_paths"][model_family]["X_train"])

        print(f"Training {model_id}...")
        if model_family == "logreg":
            model.fit(X_train, y_train)
        elif model_family == "catboost":
            model.fit(X_train, y_train, cat_features=state["nominal_categorical_columns"])
        else:
            model.fit(X_train, y_train, categorical_feature=state["nominal_categorical_columns"])

        p = artifact_dir / f"{model_id}.joblib"
        joblib.dump(model, p)
        trained_model_paths[model_id] = str(p)
        trained_model_families[model_id] = model_family

    return {
        "trained_model_paths": trained_model_paths,
        "trained_model_families": trained_model_families,
        "logs": state.get("logs", []) + ["train-models completed"],
    }



In [ ]:
def evaluate_models_node(state: PipelineState) -> Dict[str, Any]:
    y_val = joblib.load(state["split_paths"]["y_val"])
    models_dict = {model_id: joblib.load(path) for model_id, path in state["trained_model_paths"].items()}
    X_val_by_family = {
        model_family: joblib.load(paths["X_val"])
        for model_family, paths in state["model_split_paths"].items()
    }

    eval_metrics = {}
    threshold_calibrations = {}

    for model_id, model in models_dict.items():
        model_family = state["trained_model_families"][model_id]
        X_val = X_val_by_family[model_family]
        print(f"Evaluating {model_id}...")
        calibration = calibrate_threshold(model, X_val, y_val, metric=FBETA_COST_METRIC)
        threshold_calibrations[model_id] = calibration
        eval_metrics[model_id] = evaluate_binary_classifier(
            model,
            X_val,
            y_val,
            threshold=calibration["optimal_threshold"],
        )
        print(
            f"  Optimal threshold: {calibration['optimal_threshold']:.3f} "
            f"({calibration['calibration_metric_label']}={calibration['optimal_score']:.3f})"
        )

    evaluation_summary_table = []
    for model_id in models_dict:
        row = {
            "model_id": model_id,
            "val_threshold": threshold_calibrations[model_id]["optimal_threshold"],
        }
        row.update(eval_metrics[model_id])
        evaluation_summary_table.append(row)

    figures = generate_evaluation_figures(models_dict, X_val_by_family, y_val, state, threshold_calibrations)

    for model_id, calibration in threshold_calibrations.items():
        sweep_data = calibration["threshold_sweep"]
        thresholds = [d["threshold"] for d in sweep_data]
        scores = [d["score"] for d in sweep_data]

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(thresholds, scores, "o-", linewidth=2, markersize=6)
        ax.axvline(calibration["optimal_threshold"], color="red", linestyle="--", label=f"Optimal: {calibration['optimal_threshold']:.3f}")
        ax.set_xlabel("Decision Threshold")
        ax.set_ylabel(f"{calibration['calibration_metric_label']} Score")
        ax.set_title(f"Threshold Calibration - {model_id}")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        figures[f"{model_id}_threshold_sweep"] = fig

    threshold_policy = {
        "decision_thresholds": {mid: cal["optimal_threshold"] for mid, cal in threshold_calibrations.items()},
        "calibration_metric": FBETA_COST_METRIC,
        "calibration_metric_label": pretty_calibration_metric(FBETA_COST_METRIC),
        "calibration_beta": float(COST_DERIVED_BETA),
        "cost_proxy_weights": {
            "false_negative": float(FN_COST_PROXY),
            "false_positive": float(FP_COST_PROXY),
        },
        "risk_tier_thresholds": {
            "high": 0.70,
            "low": 0.00,
        },
    }

    return {
        "eval_metrics": eval_metrics,
        "threshold_calibrations": threshold_calibrations,
        "evaluation_summary_table": evaluation_summary_table,
        "figures": figures,
        "threshold_policy": threshold_policy,
        "logs": state.get("logs", []) + ["evaluate-models completed"],
    }



In [ ]:
TYPE_MAP = {
    "str": str,
    "int": int,
    "float": float,
    "bool": bool,
    "List[str]": List[str],
}


@lru_cache(maxsize=None)
def load_skill(skill_name: str, skills_dir: str) -> Dict[str, Any]:
    raw = (Path(skills_dir) / f"{skill_name}.md").read_text(encoding="utf-8")
    parts = raw.split("---", 2)
    body = parts[2].strip() if len(parts) == 3 else raw.strip()

    schema_text = body.split("## Structured Output Schema", 1)[1].split("```json", 1)[1].split("```", 1)[0].strip()
    response_schema = json.loads(schema_text)

    return {
        "body": body,
        "response_schema": response_schema,
    }


@lru_cache(maxsize=None)
def build_response_model_from_schema(schema_json: str):
    response_schema = json.loads(schema_json)
    fields = {}
    for field_name, field_spec in response_schema["fields"].items():
        fields[field_name] = (
            TYPE_MAP[field_spec["type"]],
            Field(..., description=field_spec.get("description", "")),
        )
    return create_model(response_schema["name"], **fields)


def build_response_model(response_schema: Dict[str, Any]):
    schema_json = json.dumps(response_schema, sort_keys=True)
    return build_response_model_from_schema(schema_json)


def run_llm_skill(state: PipelineState, skill_name: str, user_payload: Dict[str, Any]) -> Dict[str, Any]:
    skill = load_skill(skill_name, state["skills_dir"])
    response_model = build_response_model(skill["response_schema"])
    response = get_openai_client().responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": skill["body"]},
            {"role": "user", "content": json.dumps(user_payload, default=str, ensure_ascii=False)},
        ],
        text_format=response_model,
    )
    return response.output_parsed.model_dump()


In [ ]:


def select_model_node(state: PipelineState) -> Dict[str, Any]:
    payload = {
        "evaluation_summary_table": state["evaluation_summary_table"],
        "threshold_policy": state["threshold_policy"],
        "candidate_model_ids": list(state["trained_model_paths"].keys()),
    }

    decision = run_llm_skill(state, "select-model", payload)
    selected_model_id = decision["selected_model_id"]
    if selected_model_id not in state["trained_model_paths"]:
        raise ValueError(f"LLM returned unknown model_id: {selected_model_id}. Expected one of {list(state['trained_model_paths'].keys())}")

    return {
        "selected_model_id": selected_model_id,
        "selected_model_path": state["trained_model_paths"][selected_model_id],
        "selected_model_family": state["trained_model_families"][selected_model_id],
        "selected_model_reasoning": decision,
        "logs": state.get("logs", []) + ["select-model completed"],
    }



In [ ]:
@lru_cache(maxsize=None)
def load_cached_joblib(path: str):
    return joblib.load(path)


def compute_model_confidence(prob: float, threshold: float) -> float:
    if prob >= threshold:
        denom = max(1.0 - threshold, 1e-12)
        return float((prob - threshold) / denom)

    denom = max(threshold, 1e-12)
    return float((threshold - prob) / denom)



def confidence_band_from_score(model_confidence: float) -> str:
    if model_confidence >= 0.67:
        return "High"
    if model_confidence >= 0.34:
        return "Medium"
    return "Low"



def align_inference_features(X_df: pd.DataFrame, state: PipelineState) -> pd.DataFrame:
    X_aligned = X_df.copy()

    for col in state["feature_columns"]:
        if col not in X_aligned.columns:
            X_aligned[col] = np.nan

    extra_cols = [col for col in X_aligned.columns if col not in state["feature_columns"]]
    if extra_cols:
        X_aligned = X_aligned.drop(columns=extra_cols)

    X_aligned = X_aligned[state["feature_columns"]]

    for col in state["numeric_columns"]:
        if col in X_aligned.columns:
            X_aligned[col] = pd.to_numeric(X_aligned[col], errors="coerce")

    return X_aligned



def prepare_raw_claim_for_inference(claim_input: Dict[str, Any], state: PipelineState) -> pd.DataFrame:
    raw_claim_df = pd.DataFrame([claim_input])
    base_features = build_feature_matrix_from_raw(raw_claim_df)
    return align_inference_features(base_features, state)



def get_effective_risk_tier_thresholds(state: PipelineState) -> Dict[str, float]:
    base_thresholds = state["threshold_policy"].get(
        "risk_tier_thresholds",
        {"low": 0.0, "high": 0.70},
    )
    review_threshold = float(state["threshold_policy"]["decision_thresholds"][state["selected_model_id"]])
    high_threshold = float(max(base_thresholds.get("high", 0.70), review_threshold))
    return {
        "low": float(base_thresholds.get("low", 0.0)),
        "medium": review_threshold,
        "high": high_threshold,
    }



def run_inference_node(state: PipelineState) -> Dict[str, Any]:
    resolved_claim = state.get("inference_input") or get_demo_claim(state)

    model = load_cached_joblib(state["selected_model_path"])
    threshold = float(state["threshold_policy"]["decision_thresholds"][state["selected_model_id"]])
    risk_tier_thresholds = get_effective_risk_tier_thresholds(state)

    X_one_raw = prepare_raw_claim_for_inference(resolved_claim, state)
    X_one_ready = prepare_model_input_for_inference(X_one_raw, state, state["selected_model_family"])
    prob = float(predict_positive_probability(model, X_one_ready)[0])
    pred = int(prob >= threshold)

    decision_margin = float(abs(prob - threshold))
    model_confidence = compute_model_confidence(prob, threshold)
    confidence_band = confidence_band_from_score(model_confidence)

    if prob >= risk_tier_thresholds["high"]:
        risk_tier = "High"
    elif prob >= risk_tier_thresholds["medium"]:
        risk_tier = "Medium"
    else:
        risk_tier = "Low"

    return {
        "inference_input": resolved_claim,
        "inference_result_raw": {
            "risk_score": prob,
            "predicted_label": pred,
            "review_recommended": bool(pred),
            "threshold": threshold,
            "risk_tier": risk_tier,
            "risk_tier_thresholds": risk_tier_thresholds,
            "decision_margin": decision_margin,
            "model_confidence": model_confidence,
            "confidence_band": confidence_band,
        },
        "logs": state.get("logs", []) + ["run-inference completed"],
    }



In [ ]:


def generate_review_checklist_node(state: PipelineState) -> Dict[str, Any]:
    payload = {
        "selected_model_id": state["selected_model_id"],
        "inference_input": state["inference_input"],
        "inference_result_raw": state["inference_result_raw"],
    }

    checklist = run_llm_skill(state, "generate-review-checklist", payload)

    return {
        "review_checklist_output": checklist,
        "logs": state.get("logs", []) + ["generate-review-checklist completed"],
    }



In [ ]:


def generate_business_recommendation_node(state: PipelineState) -> Dict[str, Any]:
    payload = {
        "selected_model_id": state["selected_model_id"],
        "inference_input": state["inference_input"],
        "inference_result_raw": state["inference_result_raw"],
        "review_checklist_output": state["review_checklist_output"],
    }

    recommendation = run_llm_skill(state, "generate-business-recommendation", payload)

    return {
        "business_output": recommendation,
        "logs": state.get("logs", []) + ["generate-business-recommendation completed"],
    }



## LangGraph pipeline

In [ ]:
def load_raw_split_frame(state: PipelineState, split_name: str) -> pd.DataFrame:
    raw_split_paths = state.get("raw_split_paths", {})
    if split_name in raw_split_paths and Path(raw_split_paths[split_name]).exists():
        return load_cached_joblib(raw_split_paths[split_name]).reset_index(drop=True)

    rebuilt_splits = rebuild_raw_splits(state["raw_data_path"])
    if split_name not in rebuilt_splits:
        raise KeyError(f"Unknown raw split requested: {split_name}")
    return rebuilt_splits[split_name].reset_index(drop=True)



def get_demo_claim(state: PipelineState, case_index: int | None = None) -> Dict[str, Any]:
    X_test_raw = load_raw_split_frame(state, "X_test_raw")
    resolved_index = int(state.get("demo_case_index", 0) if case_index is None else case_index)
    if resolved_index < 0 or resolved_index >= len(X_test_raw):
        raise IndexError(f"demo_case_index {resolved_index} out of range for held-out X_test with {len(X_test_raw)} rows")
    return X_test_raw.iloc[resolved_index].to_dict()



def make_initial_state(demo_case_index: int = 0) -> PipelineState:
    return {
        "raw_data_path": str(DATA_PATH),
        "artifact_dir": str(ARTIFACT_DIR),
        "skills_dir": str(SKILLS_DIR),
        "demo_case_index": int(demo_case_index),
        "logs": [],
        "errors": [],
    }



def run_downstream_pipeline(base_state: PipelineState, claim: Dict[str, Any]) -> PipelineState:
    local_state = dict(base_state)
    local_state["logs"] = list(base_state.get("logs", []))
    local_state["errors"] = list(base_state.get("errors", []))
    local_state["inference_input"] = claim
    for node in [run_inference_node, generate_review_checklist_node, generate_business_recommendation_node]:
        local_state.update(node(local_state))
    return local_state



def run_full_notebook_pipeline(demo_case_index: int = 0) -> PipelineState:
    return app.invoke(make_initial_state(demo_case_index=demo_case_index))



def build_graph():
    graph = StateGraph(PipelineState)

    graph.add_node("preprocess-data", preprocess_data_node)
    graph.add_node("train-models", train_models_node)
    graph.add_node("evaluate-models", evaluate_models_node)
    graph.add_node("select-model", select_model_node)
    graph.add_node("run-inference", run_inference_node)
    graph.add_node("generate-review-checklist", generate_review_checklist_node)
    graph.add_node("generate-business-recommendation", generate_business_recommendation_node)

    graph.set_entry_point("preprocess-data")
    graph.add_edge("preprocess-data", "train-models")
    graph.add_edge("train-models", "evaluate-models")
    graph.add_edge("evaluate-models", "select-model")
    graph.add_edge("select-model", "run-inference")
    graph.add_edge("run-inference", "generate-review-checklist")
    graph.add_edge("generate-review-checklist", "generate-business-recommendation")
    graph.add_edge("generate-business-recommendation", END)

    return graph.compile()


app = build_graph()
print(app)



## Run Pipeline

In [ ]:
state: PipelineState = run_full_notebook_pipeline(demo_case_index=0)

## Results

In [ ]:
# Display evaluation figures and threshold calibration results
print("=" * 80)
print("EVALUATION RESULTS WITH THRESHOLD CALIBRATION")
print("=" * 80)

print("\nValidation Set Metrics (at Calibrated Thresholds):")
eval_df = pd.DataFrame(state["evaluation_summary_table"])
sort_metric = "fbeta_cost" if "fbeta_cost" in eval_df.columns else "auc_roc"
display(eval_df.sort_values(sort_metric, ascending=False))

print("\nThreshold Calibration Summary:")
for model_id, calibration in state.get("threshold_calibrations", {}).items():
    print(f"{model_id}:")
    print(f"  Optimal Threshold: {calibration['optimal_threshold']:.4f}")
    print(f"  Optimal {calibration['calibration_metric_label']} Score: {calibration['optimal_score']:.4f}")

print("\n" + "=" * 80)
print("EVALUATION FIGURES")
print("=" * 80)

for figure_name, fig in state.get("figures", {}).items():
    print(f"\n{figure_name}:")
    display(fig)



## Inspect the `select-model` decision



In [ ]:

state["selected_model_id"], state["selected_model_reasoning"]


## Held-out test evaluation for the selected model

In [ ]:

X_test = joblib.load(state["model_split_paths"][state["selected_model_family"]]["X_test"])
y_test = joblib.load(state["split_paths"]["y_test"])
selected_model = joblib.load(state["selected_model_path"])
selected_threshold = state["threshold_policy"]["decision_thresholds"][state["selected_model_id"]]

selected_test_metrics = evaluate_binary_classifier(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)
selected_test_metrics


## Inspect the end-to-end result and downstream helper




In [ ]:
demo_claim = get_demo_claim(state)
demo_state = run_downstream_pipeline(state, demo_claim)
X_test_raw_df = load_raw_split_frame(state, "X_test_raw")
test_example_idx = int(state.get("demo_case_index", 0))
example_claim = X_test_raw_df.iloc[test_example_idx].to_dict()

print(f"Notebook main graph demo case index: {test_example_idx}")
print(f"State inference_input populated by graph: {state['inference_input'] == example_claim}")
print(f"Demo claim matches selected raw test claim: {example_claim == demo_claim}")
print(f"Reusable downstream helper used explicit inference_input: {demo_state['inference_input'] == demo_claim}")

state["business_output"]



## Three end-to-end test cases

In [ ]:
def summarise_input_snapshot(claim: Dict[str, Any]) -> Dict[str, Any]:
    key_fields = [
        "months_as_customer",
        "policy_csl",
        "incident_severity",
        "total_claim_amount",
        "injury_claim",
        "property_claim",
        "vehicle_claim",
        "incident_type",
        "collision_type",
        "property_damage",
        "police_report_available",
        "authorities_contacted",
    ]
    return {key: claim.get(key) for key in key_fields if key in claim}


def risk_tier_and_priority_from_score(score: float, thresholds: Dict[str, float]) -> Dict[str, str]:
    if score >= thresholds["high"]:
        return {"expected_risk_tier": "High", "expected_priority": "P1"}
    if score >= thresholds["medium"]:
        return {"expected_risk_tier": "Medium", "expected_priority": "P2"}
    return {"expected_risk_tier": "Low", "expected_priority": "P3"}


def build_expected_output(case_kind: str, score: float, thresholds: Dict[str, float]) -> Dict[str, str]:
    expected = risk_tier_and_priority_from_score(score, thresholds)

    if case_kind == "most_suspicious_reference":
        expected.update({
            "expected_business_action": "Escalate the claim for manual fraud review and request stronger supporting evidence.",
            "expected_reason": "This is the highest-risk held-out example, so the downstream output should be decisive and operationally urgent.",
        })
    elif case_kind == "borderline_challenging":
        expected.update({
            "expected_business_action": "Use cautious, uncertainty-aware wording and recommend human review because the case sits close to the calibrated decision threshold.",
            "expected_reason": "This is the most ambiguous held-out example, so the output should acknowledge uncertainty instead of sounding over-confident.",
        })
    else:
        expected.update({
            "expected_business_action": "Allow standard processing with only routine verification steps.",
            "expected_reason": "This is the lowest-risk held-out example, so the downstream output should stay low priority unless a rule-based red flag appears.",
        })

    return expected


def run_case(case_index: int) -> Dict[str, Any]:
    claim = X_test_raw_df.iloc[int(case_index)].to_dict()
    local_state = run_downstream_pipeline(state, claim)
    inference_output = local_state["inference_result_raw"]
    business_output = local_state["business_output"]

    return {
        "case_index": int(case_index),
        "input_snapshot": summarise_input_snapshot(claim),
        "y_true": int(y_test.iloc[int(case_index)]),
        "risk_score": float(inference_output["risk_score"]),
        "decision_margin": float(inference_output["decision_margin"]),
        "model_confidence": float(inference_output["model_confidence"]),
        "confidence_band": inference_output["confidence_band"],
        "calibrated_threshold": float(inference_output["threshold"]),
        "predicted_label": int(inference_output["predicted_label"]),
        "review_recommended": bool(inference_output["review_recommended"]),
        "risk_tier": inference_output["risk_tier"],
        "actual_output": {
            "actual_risk_tier": business_output["risk_level"],
            "actual_priority": business_output["priority"],
            "actual_recommended_action": business_output["recommended_action"],
            "actual_summary_for_officer": business_output["summary_for_officer"],
        },
        "business_output": business_output,
    }


selected_model = joblib.load(state["selected_model_path"])
selected_model_id = state["selected_model_id"]
selected_model_family = state["selected_model_family"]
calibrated_threshold = float(state["threshold_policy"]["decision_thresholds"][selected_model_id])
effective_risk_tier_thresholds = get_effective_risk_tier_thresholds(state)
X_test_ready = joblib.load(state["model_split_paths"][selected_model_family]["X_test"])
y_test = joblib.load(state["split_paths"]["y_test"]).reset_index(drop=True)

X_test_scored = X_test_raw_df.copy()
X_test_scored["risk_score"] = predict_positive_probability(selected_model, X_test_ready)
X_test_scored["distance_to_threshold"] = (X_test_scored["risk_score"] - calibrated_threshold).abs()
X_test_scored["model_confidence"] = X_test_scored["risk_score"].apply(lambda prob: compute_model_confidence(float(prob), calibrated_threshold))
X_test_scored["confidence_band"] = X_test_scored["model_confidence"].apply(confidence_band_from_score)


def pick_first_unused(indexes: List[int], used: set) -> int:
    for idx in indexes:
        idx = int(idx)
        if idx not in used:
            return idx
    raise ValueError("Could not find enough distinct held-out cases for the section 13 test runs.")


used_indices = set()
top_idx = pick_first_unused(X_test_scored["risk_score"].sort_values(ascending=False).index.tolist(), used_indices)
used_indices.add(top_idx)

boundary_idx = pick_first_unused(X_test_scored["distance_to_threshold"].sort_values().index.tolist(), used_indices)
used_indices.add(boundary_idx)

low_idx = pick_first_unused(X_test_scored["risk_score"].sort_values().index.tolist(), used_indices)
used_indices.add(low_idx)

case_specs = [
    {
        "case_label": "Case 1 - Most suspicious held-out claim",
        "case_kind": "most_suspicious_reference",
        "case_index": top_idx,
    },
    {
        "case_label": "Case 2 - Borderline challenging claim",
        "case_kind": "borderline_challenging",
        "case_index": boundary_idx,
    },
    {
        "case_label": "Case 3 - Least suspicious held-out claim",
        "case_kind": "least_suspicious_reference",
        "case_index": low_idx,
    },
]

print("Running 3 end-to-end test cases for the report:")
print(f"  Case 1 index={top_idx} | risk_score={X_test_scored.loc[top_idx, 'risk_score']:.4f} | model_confidence={X_test_scored.loc[top_idx, 'model_confidence']:.4f}")
print(f"  Case 2 index={boundary_idx} | risk_score={X_test_scored.loc[boundary_idx, 'risk_score']:.4f} | model_confidence={X_test_scored.loc[boundary_idx, 'model_confidence']:.4f}")
print(f"  Case 3 index={low_idx} | risk_score={X_test_scored.loc[low_idx, 'risk_score']:.4f} | model_confidence={X_test_scored.loc[low_idx, 'model_confidence']:.4f}")

case_runs = []
for spec in case_specs:
    actual = run_case(spec["case_index"])
    expected = build_expected_output(spec["case_kind"], actual["risk_score"], effective_risk_tier_thresholds)
    case_runs.append({
        "case_label": spec["case_label"],
        "case_index": actual["case_index"],
        "input_snapshot": actual["input_snapshot"],
        "expected_output": expected,
        "actual_output": actual["actual_output"],
        "y_true": actual["y_true"],
        "ground_truth_label": "Fraud" if actual["y_true"] == 1 else "Legitimate",
        "risk_score": actual["risk_score"],
        "decision_margin": actual["decision_margin"],
        "model_confidence": actual["model_confidence"],
        "confidence_band": actual["confidence_band"],
        "calibrated_threshold": actual["calibrated_threshold"],
        "review_recommended": actual["review_recommended"],
        "predicted_label": actual["predicted_label"],
        "risk_tier": actual["risk_tier"],
        "business_output": actual["business_output"],
    })

for run in case_runs:
    print("\n" + "=" * 80)
    print(run["case_label"])
    print("=" * 80)
    print(f"Held-out case index: {run['case_index']}")
    print(f"Ground truth: {run['ground_truth_label']} (y_true={run['y_true']})")
    print(f"risk_score={run['risk_score']:.4f} | threshold={run['calibrated_threshold']:.4f} | review_recommended={run['review_recommended']} | confidence={run['model_confidence']:.4f}")
    print("\nInput snapshot:")
    display(pd.DataFrame([run["input_snapshot"]]))
    print("\nExpected business-facing output:")
    display(pd.DataFrame([run["expected_output"]]))
    print("\nActual downstream output:")
    display(pd.DataFrame([run["actual_output"]]))



## Failures


In [ ]:

# Evidence table for the "What Went Wrong and Why" section
X_test_raw = joblib.load(state["split_paths"]["X_test"])
X_test_ready = joblib.load(state["model_split_paths"][state["selected_model_family"]]["X_test"])
y_test = joblib.load(state["split_paths"]["y_test"])
selected_model = joblib.load(state["selected_model_path"])

calibrated_threshold = state["threshold_policy"]["decision_thresholds"][state["selected_model_id"]]
baseline_threshold = 0.5
prob_test = predict_positive_probability(selected_model, X_test_ready)
pred_test_default = (prob_test >= baseline_threshold).astype(int)
pred_test_calibrated = (prob_test >= calibrated_threshold).astype(int)

errors_df = X_test_raw.copy()
errors_df["y_true"] = y_test.values
errors_df["y_pred_default"] = pred_test_default
errors_df["y_pred_calibrated"] = pred_test_calibrated
errors_df["risk_score"] = prob_test
errors_df["distance_to_threshold"] = np.abs(errors_df["risk_score"] - calibrated_threshold)
errors_df["model_confidence"] = [compute_model_confidence(float(prob), calibrated_threshold) for prob in prob_test]
errors_df["confidence_band"] = errors_df["model_confidence"].apply(confidence_band_from_score)
errors_df["is_error_default"] = (errors_df["y_true"] != errors_df["y_pred_default"]).astype(int)
errors_df["is_error_calibrated"] = (errors_df["y_true"] != errors_df["y_pred_calibrated"]).astype(int)
errors_df["error_type_calibrated"] = np.select(
    [
        (errors_df["y_true"] == 0) & (errors_df["y_pred_calibrated"] == 1),
        (errors_df["y_true"] == 1) & (errors_df["y_pred_calibrated"] == 0),
    ],
    ["False Positive", "False Negative"],
    default="Correct",
)

summary_df = pd.DataFrame([
    {
        "threshold_name": "Fixed baseline",
        "threshold": baseline_threshold,
        "total_errors": int(errors_df["is_error_default"].sum()),
        "false_positives": int(((errors_df["y_true"] == 0) & (errors_df["y_pred_default"] == 1)).sum()),
        "false_negatives": int(((errors_df["y_true"] == 1) & (errors_df["y_pred_default"] == 0)).sum()),
    },
    {
        "threshold_name": "Calibrated",
        "threshold": calibrated_threshold,
        "total_errors": int(errors_df["is_error_calibrated"].sum()),
        "false_positives": int(((errors_df["y_true"] == 0) & (errors_df["y_pred_calibrated"] == 1)).sum()),
        "false_negatives": int(((errors_df["y_true"] == 1) & (errors_df["y_pred_calibrated"] == 0)).sum()),
    },
])

print("=" * 80)
print("WHAT WENT WRONG AND WHY: FAILURE EVIDENCE")
print("=" * 80)
print(f"Selected model: {state['selected_model_id']}")
print(f"Calibrated threshold: {calibrated_threshold:.4f}\n")

print("Threshold comparison summary:")
display(summary_df)

print("\nTop calibrated false positives:")
display(
    errors_df[errors_df["error_type_calibrated"] == "False Positive"]
    .sort_values("risk_score", ascending=False)
    .head(5)
)

print("\nTop calibrated false negatives:")
display(
    errors_df[errors_df["error_type_calibrated"] == "False Negative"]
    .sort_values("risk_score", ascending=True)
    .head(5)
)

print("\nClosest cases to the calibrated threshold (useful for analysing low-confidence outputs):")
display(
    errors_df.sort_values("distance_to_threshold", ascending=True)
    [["risk_score", "model_confidence", "confidence_band", "distance_to_threshold", "y_true", "y_pred_calibrated", "error_type_calibrated"]]
    .head(10)
)


## Gradio interface


In [ ]:
print("Legacy minimal Gradio demo removed. Use the full form-based interface below.")


In [ ]:
import time
import gradio as gr
import joblib
import pandas as pd

SCENARIO_LABELS = [
    "Scenario A: High-Risk Suspicious Claim (High Confidence)",
    "Scenario B: Borderline Ambiguous Claim (Low Confidence)",
    "Scenario C: Preferred Customer Routine Claim (Low Risk)",
]

SECTION_META = [
    {
        "section": "Policy",
        "step": "01",
        "title": "Policy Context",
        "description": "Establish the policy history, coverage structure, and financial baseline before scoring the claim.",
        "fields": [
            "months_as_customer",
            "age",
            "policy_bind_date",
            "policy_state",
            "policy_csl",
            "policy_deductable",
            "policy_annual_premium",
            "umbrella_limit",
        ],
    },
    {
        "section": "Insured",
        "step": "02",
        "title": "Insured Profile",
        "description": "Capture the customer profile and declared background signals that may contextualize the incident.",
        "fields": [
            "insured_sex",
            "insured_education_level",
            "insured_occupation",
            "insured_hobbies",
            "insured_relationship",
            "capital-gains",
            "capital-loss",
        ],
    },
    {
        "section": "Incident",
        "step": "03",
        "title": "Incident Details",
        "description": "Describe how the event happened, who was involved, and what verification evidence is available.",
        "fields": [
            "incident_date",
            "incident_type",
            "collision_type",
            "incident_severity",
            "authorities_contacted",
            "incident_state",
            "incident_city",
            "incident_hour_of_the_day",
            "number_of_vehicles_involved",
            "property_damage",
            "bodily_injuries",
            "witnesses",
            "police_report_available",
        ],
    },
    {
        "section": "Claim Amounts",
        "step": "04",
        "title": "Claim Breakdown",
        "description": "Enter the financial exposure and the split across bodily injury, property, and vehicle components.",
        "fields": [
            "total_claim_amount",
            "injury_claim",
            "property_claim",
            "vehicle_claim",
        ],
    },
    {
        "section": "Vehicle",
        "step": "05",
        "title": "Vehicle Information",
        "description": "Add the vehicle details used to complete the claim record before sending it to the triage pipeline.",
        "fields": [
            "auto_make",
            "auto_model",
            "auto_year",
        ],
    },
]

UI_FIELD_SECTIONS = [(meta["section"], meta["fields"]) for meta in SECTION_META]


In [ ]:
def ensure_base_state():
    if "state" in globals() and isinstance(state, dict) and state.get("selected_model_path"):
        return state

    if "app" not in globals() or "make_initial_state" not in globals():
        raise RuntimeError(
            "LangGraph app/state is not ready. Please run the upstream pipeline cells first "
            "(build_graph + app.invoke)."
        )

    generated_state = app.invoke(make_initial_state(demo_case_index=0))
    globals()["state"] = generated_state
    return generated_state


def get_raw_claim_reference_frame(base_state: PipelineState) -> pd.DataFrame:
    return load_raw_split_frame(base_state, "X_test_raw")


def get_raw_feature_reference(base_state: PipelineState) -> pd.DataFrame:
    raw_df = pd.read_csv(base_state["raw_data_path"]).copy()
    return raw_df.drop(columns=[TARGET_COLUMN]).copy()


def pick_case_indices(base_state: PipelineState):
    raw_x_test_df = get_raw_claim_reference_frame(base_state)
    selected_model = joblib.load(base_state["selected_model_path"])
    selected_family = base_state["selected_model_family"]
    threshold = float(base_state["threshold_policy"]["decision_thresholds"][base_state["selected_model_id"]])

    x_test_ready = joblib.load(base_state["model_split_paths"][selected_family]["X_test"])
    scores = predict_positive_probability(selected_model, x_test_ready)

    scored = pd.DataFrame({"risk_score": scores})
    scored["distance_to_threshold"] = (scored["risk_score"] - threshold).abs()

    used = set()

    def first_unused(indices):
        for i in indices:
            idx = int(i)
            if idx not in used:
                return idx
        raise RuntimeError("Unable to pick three distinct held-out cases.")

    high_idx = first_unused(scored["risk_score"].sort_values(ascending=False).index.tolist())
    used.add(high_idx)

    boundary_idx = first_unused(scored["distance_to_threshold"].sort_values().index.tolist())
    used.add(boundary_idx)

    low_idx = first_unused(scored["risk_score"].sort_values().index.tolist())
    used.add(low_idx)

    scenario_to_idx = {
        SCENARIO_LABELS[0]: high_idx,
        SCENARIO_LABELS[1]: boundary_idx,
        SCENARIO_LABELS[2]: low_idx,
    }
    return raw_x_test_df, scenario_to_idx


def build_ui_field_specs(base_state: PipelineState) -> List[Dict[str, Any]]:
    raw_reference = get_raw_feature_reference(base_state)
    raw_reference = raw_reference.replace(list(MISSING_CATEGORY_TOKENS), pd.NA)
    specs = []

    for section_name, fields in UI_FIELD_SECTIONS:
        for field_name in fields:
            series = raw_reference[field_name]
            clean = series.dropna()

            spec = {
                "section": section_name,
                "name": field_name,
                "label": field_name.replace("_", " ").title(),
            }

            if field_name in {"policy_bind_date", "incident_date"}:
                spec["kind"] = "text"
                spec["placeholder"] = "YYYY-MM-DD"
                spec["default"] = str(clean.astype(str).mode().iloc[0]) if not clean.empty else ""
            elif pd.api.types.is_numeric_dtype(series):
                spec["kind"] = "number"
                median = clean.median() if not clean.empty else 0
                if pd.api.types.is_integer_dtype(series):
                    median = int(round(float(median)))
                spec["default"] = median
            else:
                choices = sorted(clean.astype(str).unique().tolist())
                spec["choices"] = choices
                spec["default"] = clean.astype(str).mode().iloc[0] if not clean.empty else None
                if len(choices) == 2:
                    spec["kind"] = "radio"
                elif len(choices) <= 25:
                    spec["kind"] = "dropdown"
                else:
                    spec["kind"] = "text"
                    spec["placeholder"] = "Enter value"

            specs.append(spec)

    return specs



In [ ]:
def build_risk_meter_html(score, threshold, risk_tier_thresholds):
    score_pct = max(0.0, min(100.0, float(score) * 100.0))
    threshold_pct = max(0.0, min(100.0, float(threshold) * 100.0))
    medium_pct = max(0.0, min(100.0, float(risk_tier_thresholds["medium"]) * 100.0))
    high_pct = max(medium_pct, min(100.0, float(risk_tier_thresholds["high"]) * 100.0))

    medium_width = max(0.0, high_pct - medium_pct)
    high_width = max(0.0, 100.0 - high_pct)

    return f"""
    <div class="meter-shell">
      <div class="meter-summary">
        <div><strong>{score_pct:.1f}%</strong><span>Fraud probability</span></div>
        <div><strong>{threshold_pct:.1f}%</strong><span>Review threshold</span></div>
      </div>
      <div class="meter-track">
        <div class="meter-zone zone-low" style="width: {medium_pct:.2f}%"></div>
        <div class="meter-zone zone-medium" style="left: {medium_pct:.2f}%; width: {medium_width:.2f}%"></div>
        <div class="meter-zone zone-high" style="left: {high_pct:.2f}%; width: {high_width:.2f}%"></div>
        <div class="meter-threshold" style="left: {threshold_pct:.2f}%"></div>
        <div class="meter-score" style="left: {score_pct:.2f}%"></div>
      </div>
      <div class="meter-scale">
        <span>Low</span>
        <span>Medium</span>
        <span>High</span>
      </div>
    </div>
    """


def normalize_form_value(value):
    if pd.isna(value):
        return None
    if value in ("", "MISSING", None):
        return None
    if isinstance(value, str) and value.strip() in {"", "?", "MISSING"}:
        return None
    return value


def coerce_claim_payload(field_names: List[str], values: List[Any]) -> Dict[str, Any]:
    claim = {}
    for field_name, value in zip(field_names, values):
        claim[field_name] = normalize_form_value(value)
    return claim


def format_claim_for_form(claim: Dict[str, Any], field_specs: List[Dict[str, Any]]) -> List[Any]:
    values = []
    for spec in field_specs:
        value = normalize_form_value(claim.get(spec["name"], spec.get("default")))

        if value is None:
            values.append(None)
            continue

        if spec["kind"] == "number":
            value = float(value)
            if float(value).is_integer():
                value = int(value)
        else:
            value = str(value)
            if spec["kind"] in {"radio", "dropdown"} and value not in spec.get("choices", []):
                value = None

        values.append(value)
    return values


def strip_technical_trace(text: Any) -> str:
    if text is None:
        return "N/A"

    cleaned = str(text)
    marker = "Technical Trace"
    if marker in cleaned:
        cleaned = cleaned.split(marker, 1)[0].rstrip()
    return cleaned.strip() or "N/A"


def build_business_markdown(source_label: str, local_state: dict) -> str:
    business_output = local_state["business_output"]
    raw = local_state["inference_result_raw"]
    checklist = local_state.get("review_checklist_output", {}).get("checklist", [])
    checklist_md = "\n".join(f"- {item}" for item in checklist) if checklist else "- No checklist items generated"

    technical_trace = "\n".join([
        "### Technical Trace",
        f"- **Selected Model:** {local_state.get('selected_model_id', 'N/A')}",
        f"- **Risk Score:** {float(raw.get('risk_score', 0.0)):.4f}",
        f"- **Model Confidence:** {float(raw.get('model_confidence', 0.0)):.4f}",
        f"- **Threshold:** {float(raw.get('threshold', 0.0)):.4f}",
        f"- **Predicted Label:** {int(raw.get('predicted_label', 0))}",
        f"- **Risk Tier:** {raw.get('risk_tier', 'N/A')}",
        f"- **Decision Margin:** {float(raw.get('decision_margin', 0.0)):.4f}",
        f"- **Confidence Band:** {raw.get('confidence_band', 'N/A')}",
    ])

    return f"""## {business_output.get('title', 'Insurance Fraud Risk Triage Report')}

**Input Source:** {source_label}

### Decision Summary
- **Risk Level:** {business_output.get('risk_level', 'N/A')}
- **Priority:** {business_output.get('priority', 'N/A')}
- **Recommended Action:** {business_output.get('recommended_action', 'N/A')}

### Officer Summary
{strip_technical_trace(business_output.get('summary_for_officer', 'N/A'))}

### Review Checklist
{checklist_md}

### Customer Next Step Message
{strip_technical_trace(business_output.get('customer_next_step_message', 'N/A'))}

### Compliance Note
{strip_technical_trace(business_output.get('disclaimer', 'N/A'))}

{technical_trace}
"""


def load_template_into_form(template_label: str):
    if template_label not in PRESET_FORM_VALUES:
        raise ValueError(f"Unknown template label: {template_label}")

    return list(PRESET_FORM_VALUES[template_label])


def run_pipeline_demo(source_label: str, *form_values):
    try:
        claim = coerce_claim_payload(UI_FIELD_NAMES, list(form_values))
        local_state = run_downstream_pipeline(BASE_STATE_FOR_DEMO, claim)
        raw = local_state["inference_result_raw"]

        meter_html = build_risk_meter_html(
            raw["risk_score"],
            raw["threshold"],
            raw["risk_tier_thresholds"],
        )

        final_report = build_business_markdown(source_label, local_state)
        return meter_html, final_report

    except Exception as exc:
        return (
            "<div class='meter-shell meter-empty'><strong>No score available.</strong><span>Resolve the input issue and retry.</span></div>",
            f"### Execution Error\n\n{exc}",
        )



In [ ]:
custom_css = """
:root {
    --bg: #f4ede3;
    --panel: #fffdf9;
    --card: #ffffff;
    --ink: #1f2833;
    --muted: #6f726d;
    --line: #e4d7c7;
    --accent: #1f5a44;
    --accent-deep: #173f31;
    --accent-soft: #e5efe9;
    --shadow: 0 14px 34px rgba(47, 37, 28, 0.08);
    --serif: "Iowan Old Style", "Palatino Linotype", "Book Antiqua", Georgia, serif;
    --sans: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}

html,
body,
.gradio-container,
.gradio-container * {
    color-scheme: light !important;
}

body,
.gradio-container,
.markdown-text,
span,
p,
div,
label,
input,
textarea,
select,
button {
    font-family: var(--sans) !important;
    color: var(--ink);
}

html,
body,
.gradio-container,
.dark,
.dark body,
.dark .gradio-container {
    background: linear-gradient(180deg, #f8f3ec 0%, #f3eadf 100%) !important;
    color: var(--ink) !important;
}

.gradio-container {
    background: linear-gradient(180deg, #f8f3ec 0%, #f3eadf 100%);
    padding: 18px 16px 32px 16px !important;
}

input,
textarea,
select,
button,
.dark input,
.dark textarea,
.dark select,
.dark button {
    background-color: #fffdf9 !important;
    color: var(--ink) !important;
    border-color: var(--line) !important;
}

.gradio-container .gr-markdown,
.gradio-container .prose,
.gradio-container .gr-prose {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    padding: 0 !important;
}

.results-shell .gr-markdown,
.results-shell .gr-markdown > div,
.results-shell .gr-prose,
.results-shell .prose,
.results-shell .gr-html,
.results-shell .gr-html > div,
.results-shell [class*="wrap"],
.results-shell [class*="block"],
.results-shell [class*="container"],
.results-shell [class*="group"] {
    background: transparent !important;
    box-shadow: none !important;
    border: none !important;
}

.hero-shell {
    background: linear-gradient(135deg, rgba(15, 28, 24, 0.96), rgba(27, 63, 49, 0.94));
    border-radius: 28px;
    padding: 30px 30px 26px 30px;
    color: #f8f3ec;
    box-shadow: 0 18px 44px rgba(26, 32, 35, 0.16);
}

.hero-kicker {
    letter-spacing: 0.16em;
    text-transform: uppercase;
    font-size: 11px;
    opacity: 0.76;
    margin-bottom: 10px;
}

.hero-title,
.section-heading h3,
.review-header h2,
.form-toolbar-copy strong,
.control-heading strong,
.guide-item strong {
    font-family: var(--serif) !important;
}

.hero-title {
    font-size: clamp(32px, 4vw, 54px);
    line-height: 0.98;
    margin: 0;
    max-width: none;
    width: 100%;
    color: #fff8ef;
}

.guide-shell,
.guide-shell > div,
.control-shell,
.control-shell > div,
.survey-shell,
.survey-shell > div,
.results-shell,
.results-shell > div,
.results-shell > div > div {
    background: var(--panel) !important;
    border: none !important;
    border-radius: 26px !important;
    box-shadow: none !important;
}

.guide-shell,
.control-shell,
.survey-shell,
.results-shell {
    padding: 18px;
}

.content-safe-pad,
.guide-item,
.control-heading,
.form-toolbar-copy,
.section-heading,
.review-header,
.question-card,
.meter-shell,
.business-report,
.business-report > div,
.business-report [data-testid="markdown"],
.business-report [data-testid="markdown"] > div,
.business-report .gr-markdown,
.business-report .gr-markdown > div,
.business-report .gr-prose,
.business-report .prose {
    box-sizing: border-box !important;
    padding-left: 5px !important;
    padding-right: 5px !important;
}

.guide-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 12px;
}

.guide-item {
    background: #fffdf9 !important;
    border: 1px solid var(--line);
    border-radius: 18px;
    padding: 16px 18px;
}

.guide-item strong {
    display: block;
    font-size: 20px;
    margin-bottom: 6px;
    color: var(--accent-deep);
}

.guide-item span,
.control-heading span,
.form-toolbar-copy span,
.section-heading p {
    display: block;
    color: var(--muted);
    font-size: 14px;
    line-height: 1.65;
}

.control-heading {
    margin-bottom: 12px;
}

.control-heading strong,
.form-toolbar-copy strong {
    display: block;
    font-size: 24px;
    line-height: 1.1;
    margin-bottom: 6px;
    color: var(--ink);
}

.form-toolbar {
    align-items: end;
    gap: 16px;
    margin-bottom: 12px;
}

.step-strip {
    display: grid;
    grid-template-columns: repeat(5, minmax(0, 1fr));
    gap: 10px;
    margin: 14px 0 8px 0;
}

.step-pill {
    padding: 10px 12px;
    border-radius: 16px;
    background: #fffdf9 !important;
    border: 1px solid var(--line);
    min-height: 62px;
}

.step-pill strong {
    display: block;
    font-size: 11px;
    letter-spacing: 0.1em;
    text-transform: uppercase;
    color: var(--accent);
    margin-bottom: 4px;
}

.step-pill span {
    display: block;
    font-size: 13px;
    line-height: 1.35;
    color: var(--ink);
}

.questionnaire-tabs button {
    border-radius: 999px !important;
    border: 1px solid var(--line) !important;
    background: rgba(255, 255, 255, 0.88) !important;
    color: var(--ink) !important;
    font-weight: 600 !important;
    padding: 10px 16px !important;
}

.questionnaire-tabs button.selected {
    background: linear-gradient(135deg, var(--accent), var(--accent-deep)) !important;
    color: #fff !important;
    border-color: transparent !important;
}

.section-heading {
    margin: 6px 0 12px 0;
}

.section-heading h3 {
    margin: 0 0 6px 0;
    font-size: 26px;
    color: var(--ink);
}

.question-card {
    background: #fffdf9 !important;
    border: 1px solid var(--line);
    border-radius: 18px;
    padding: 6px 8px 2px 8px;
    min-height: 92px;
    box-shadow: none !important;
}

.question-card label span {
    font-size: 13px !important;
    letter-spacing: 0.01em;
}

.preset-action button,
.submit-action button {
    height: 50px !important;
    border-radius: 999px !important;
    font-weight: 700 !important;
    font-size: 14px !important;
    box-shadow: none !important;
}

.preset-action button {
    background: linear-gradient(135deg, #2e6d54, var(--accent-deep)) !important;
    border: none !important;
    color: #fff !important;
    box-shadow: 0 10px 24px rgba(23, 63, 49, 0.18) !important;
}

.submit-action button {
    background: linear-gradient(135deg, var(--accent), var(--accent-deep)) !important;
    border: none !important;
    color: #fff !important;
    box-shadow: 0 14px 28px rgba(23, 63, 49, 0.22) !important;
}

.review-header {
    margin-bottom: 14px;
}

.review-header span {
    display: inline-block;
    font-size: 11px;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: var(--accent);
    margin-bottom: 8px;
}

.review-header h2 {
    margin: 0;
    font-size: 34px;
    line-height: 1.02;
    color: var(--ink);
}

.review-stack {
    display: flex;
    flex-direction: column;
    gap: 16px;
    background: transparent !important;
    box-shadow: none !important;
    border: none !important;
}

.review-stack > div,
.review-stack [class*="wrap"],
.review-stack [class*="block"],
.review-stack [class*="container"],
.review-stack [class*="group"] {
    background: transparent !important;
    background-color: transparent !important;
    box-shadow: none !important;
    border: none !important;
}

.business-report,
.business-report > div,
.business-report [data-testid="markdown"],
.business-report [data-testid="markdown"] > div,
.business-report .gr-markdown,
.business-report .gr-markdown > div,
.business-report .gr-prose,
.business-report .prose,
.business-report .prose *,
.business-report [class*="wrap"],
.business-report [class*="block"],
.business-report [class*="container"] {
    background: transparent !important;
    background-color: transparent !important;
    box-shadow: none !important;
    border: none !important;
}

.meter-shell {
    background: #fffdf9 !important;
    border: 1px solid var(--line);
    border-radius: 20px;
    padding: 14px;
    box-shadow: none !important;
}

.meter-summary {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 10px;
    margin-bottom: 12px;
}

.meter-summary strong {
    display: block;
    font-size: 24px;
    line-height: 1;
    color: var(--accent-deep);
}

.meter-summary span {
    font-size: 12px;
    color: var(--muted);
}

.meter-track {
    position: relative;
    height: 16px;
    border-radius: 999px;
    background: #efe7dc;
    overflow: hidden;
    border: 1px solid rgba(228, 215, 199, 0.9);
}

.meter-zone {
    position: absolute;
    top: 0;
    bottom: 0;
}

.zone-low {
    left: 0;
    background: linear-gradient(90deg, #bfe7c8, #83d7a0);
}

.zone-medium {
    background: linear-gradient(90deg, #f8d785, #f1bf54);
}

.zone-high {
    background: linear-gradient(90deg, #f7aa93, #ea6b56);
}

.meter-threshold,
.meter-score {
    position: absolute;
    top: -5px;
    width: 2px;
    height: 24px;
    transform: translateX(-50%);
    z-index: 3;
}

.meter-threshold {
    background: #16342a;
}

.meter-score {
    background: #204fd6;
}

.meter-score::after {
    content: "";
    position: absolute;
    top: -5px;
    left: 50%;
    width: 12px;
    height: 12px;
    transform: translate(-50%, 0);
    border-radius: 50%;
    background: #204fd6;
    border: 2px solid #fff;
    box-shadow: 0 4px 10px rgba(32, 79, 214, 0.18);
}

.meter-scale {
    display: flex;
    justify-content: space-between;
    margin-top: 10px;
    font-size: 12px;
    color: var(--muted);
}

.meter-empty {
    display: flex;
    flex-direction: column;
    gap: 4px;
}

@media (max-width: 900px) {
    .guide-grid,
    .step-strip,
    .meter-summary {
        grid-template-columns: 1fr;
    }
}
"""

BASE_STATE_FOR_DEMO = ensure_base_state()
UI_FIELD_SPECS = build_ui_field_specs(BASE_STATE_FOR_DEMO)
FIELD_SPEC_MAP = {spec["name"]: spec for spec in UI_FIELD_SPECS}
UI_FIELD_NAMES = [spec["name"] for spec in UI_FIELD_SPECS]
RAW_X_TEST_REFERENCE, PRESET_SCENARIO_TO_IDX = pick_case_indices(BASE_STATE_FOR_DEMO)
PRESET_FORM_VALUES = {
    label: format_claim_for_form(RAW_X_TEST_REFERENCE.iloc[int(case_index)].to_dict(), UI_FIELD_SPECS)
    for label, case_index in PRESET_SCENARIO_TO_IDX.items()
}

input_components = []

hero_html = """
<div class="hero-shell">
  <div class="hero-kicker">Insurance Fraud Intake</div>
  <div class="hero-title">Fraud Traige System</div>
</div>
"""

guide_html = """
<div class="guide-grid">
  <div class="guide-item">
    <strong>5 steps</strong>
    <span>Fill in all five sections below: Policy, Insured, Incident, Claim Amounts, and Vehicle</span>
  </div>
  <div class="guide-item">
    <strong>Preset paths</strong>
    <span>You can enter a claim manually or apply a preset template first for testing, then edit any field before submission</span>
  </div>
</div>
"""

preset_heading_html = """
<div class="control-heading">
  <strong>Preset template</strong>
  <span>Select a prepared scenario on the left and apply it into the form on the right</span>
</div>
"""

form_toolbar_html = """
<div class="form-toolbar-copy">
  <strong>Claim details</strong>
  <span>Complete the five sections below and submit the claim when the form is ready</span>
</div>
"""

review_header_html = """
<div class="review-header">
  <span>Decision Layer</span>
  <h2>Review outcome and business recommendation</h2>
</div>
"""

with gr.Blocks(title="Insurance Fraud Risk Triage Demo", theme=gr.themes.Soft(), css=custom_css) as triage_demo:
    gr.HTML(hero_html)

    with gr.Group(elem_classes=["guide-shell"]):
        gr.HTML(guide_html)

    with gr.Group(elem_classes=["control-shell"]):
        gr.HTML(preset_heading_html)
        with gr.Row(equal_height=True):
            template_selector = gr.Dropdown(
                choices=SCENARIO_LABELS,
                value=SCENARIO_LABELS[0],
                label="Choose a preset scenario",
                scale=8,
            )
            load_template_button = gr.Button("Apply To Form", elem_classes=["preset-action"], scale=2)

    with gr.Group(elem_classes=["survey-shell"]):
        with gr.Row(equal_height=False, elem_classes=["form-toolbar"]):
            with gr.Column(scale=10):
                gr.HTML(form_toolbar_html)
            with gr.Column(scale=2, min_width=180):
                run_button = gr.Button("Submit Claim", variant="primary", elem_classes=["submit-action"])

        with gr.Tabs(elem_classes=["questionnaire-tabs"]):
            for meta in SECTION_META:
                with gr.Tab(f"{meta['step']} {meta['title']}"):
                    gr.HTML(
                        f"<div class='section-heading'><h3>{meta['title']}</h3><p>{meta['description']}</p></div>"
                    )

                    section_specs = [FIELD_SPEC_MAP[name] for name in meta["fields"]]
                    for row_start in range(0, len(section_specs), 2):
                        row_specs = section_specs[row_start: row_start + 2]
                        with gr.Row():
                            for spec in row_specs:
                                with gr.Column():
                                    with gr.Group(elem_classes=["question-card"]):
                                        if spec["kind"] == "radio":
                                            component = gr.Radio(
                                                choices=spec["choices"],
                                                value=spec.get("default"),
                                                label=spec["label"],
                                            )
                                        elif spec["kind"] == "dropdown":
                                            component = gr.Dropdown(
                                                choices=spec["choices"],
                                                value=spec.get("default"),
                                                label=spec["label"],
                                                allow_custom_value=False,
                                            )
                                        elif spec["kind"] == "number":
                                            component = gr.Number(
                                                value=spec.get("default"),
                                                label=spec["label"],
                                            )
                                        else:
                                            component = gr.Textbox(
                                                value=spec.get("default", ""),
                                                label=spec["label"],
                                                placeholder=spec.get("placeholder", ""),
                                            )

                                    input_components.append(component)

    with gr.Group(elem_classes=["results-shell"]):
        gr.HTML(review_header_html)
        with gr.Group(elem_classes=["review-stack"]):
            risk_meter_html = gr.HTML("<div class='meter-shell meter-empty'><strong>No score yet.</strong><span>Submit a claim to render the risk meter.</span></div>")
            business_report_md = gr.Markdown("Submit a claim to generate the business-facing decision.", elem_classes=["business-report"])

    load_template_button.click(
        fn=load_template_into_form,
        inputs=[template_selector],
        outputs=input_components,
    )

    run_button.click(
        fn=lambda *values: run_pipeline_demo("Questionnaire Submission", *values),
        inputs=input_components,
        outputs=[risk_meter_html, business_report_md],
    )

triage_demo.launch(share=True, inline=True, debug=False)
print("Questionnaire-style Gradio UI launched with share=True.")

